# Setup

In [1]:
# Standard library imports
import os
import subprocess
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

# Third-party imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import torch
import xarray as xr
from osgeo import gdal
from tqdm import tqdm

%matplotlib inline

In [2]:
# !pip install --upgrade transformers
# !pip install tiler

In [3]:
from transformers import AutoImageProcessor
from tiler import Tiler, Merger

In [4]:
repo_dir = "lfm"

sys.path.append("lfm")

from lfm.tasks.sem_segmentation.dataset import get_dataloaders, calculate_dataset_statistics
# from lfm.tasks.sem_segmentation.driver import train_model, prepare_image_for_display, convert_binary_masks_to_instance_map
from lfm.tasks.sem_segmentation.model import load_dinov3_encoder, DINOSegmentation

# Config

In [5]:
# Data paths
INPUT_DIR = "/explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/data_cubes/5_band_WAC/"

# Where to load dinov3 init weights
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'
pretrained_checkpoint = "/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/checkpoint_epoch_050.pt"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs_meeting"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 8

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Output directory: ./outputs_meeting
Using device: cuda


# Load model

In [6]:
# Load model from model factory
encoder = load_dinov3_encoder(weights_local_checkpoint, device) 
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE
).to(device)

# Apply checkpoint
checkpoint = torch.load(pretrained_checkpoint, map_location='cpu')
checkpoint_state = checkpoint['model_state_dict']
missing_keys, unexpected_keys = model.load_state_dict(checkpoint_state, strict=False)
model.to(device)
model.eval()
print("Successfully loaded model from checkpoint!")

Loading model from /explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth


Using cache found in /home/ajkerr1/.cache/torch/hub/facebookresearch_dinov3_main


Encoder loaded with pretrained weights.


/explore/nobackup/people/ajkerr1/.nccstmp/ipykernel_503445/1494035847.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pretrained_checkpoint, map

Successfully loaded model from checkpoint!


# Load data

## Load statistics from training to normalize inputs

In [7]:
print("\n" + "="*60)
print("STEP 1: Loading training dataset statistics.")
print("="*60)

# Load mean and std from a previous training run
MEAN = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_mean.npy")
STD = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_std.npy")
print(MEAN, STD)


STEP 1: Loading training dataset statistics.
[0.32744208 0.32249309 0.302985  ] [0.15045737 0.15014801 0.14386101]


## Load geotiffs from path

In [8]:
# tiffs = glob(f"{tiff_dir}/*.tif")
# print(f"Found {len(tiffs)} TIFF files")
# if len(tiffs) == 0:
#     raise ValueError(f"No TIFF files found in {tiff_dir}")

# Plot data cubes

In [9]:
def plot_data(tiffs, n_samples, n_bands, output_dir):
    samples = tiffs[:n_samples]
    images = []
    filenames = []

    now = datetime.now()
    time_str = now.strftime("%y%m%d_%H%M")
    output_basename = f"{time_str}"


    print(f"Loading {len(samples)} TIFF files...")
    for tiff_path in samples:
        dataset = gdal.Open(tiff_path)
        ds_band_count = dataset.RasterCount
        base_tiff_path = Path(tiff_path).name

        print(f"Opened {base_tiff_path} with {n_bands} bands")

        # Read first n_bands (1-indexed in GDAL)
        band_list = list(range(1, min(n_bands, ds_band_count) + 1))
        img_array = dataset.ReadAsArray(band_list=band_list)

        # Track mask changes
        current_group_start = 0
        mask_change_bands = []

        print("\n" + "="*60)
        print("MASK CHANGE DETECTION")
        print("="*60)

        for i in range(0, n_bands, 25):
            fig, axes = plt.subplots(5, 5, figsize=(10, 10))
            bands_subset = img_array[i:i+25]
            band_idx = 0

            for row in range(5):
                for col in range(5):
                    current_band_num = band_idx + i

                    if band_idx < len(bands_subset) and current_band_num < n_bands:
                        band_to_plot = bands_subset[band_idx]
                        band_min, band_max = np.min(band_to_plot), np.max(band_to_plot)
                        print("Unmasked band min, max:", band_min, band_max)

                        # Mask out nodata values
                        mask = (band_to_plot == -3.4028227e+38) # Nodata value is large negative number
                        print(f"Number of masked values: {mask.sum()}")
                        masked_band = ma.masked_where(mask, band_to_plot)
                        valid_values = masked_band.compressed()
                        if valid_values.size > 0:
                            min_val = valid_values.min()
                            max_val = valid_values.max()
                            print(f"Range of unmasked values: [{min_val}, {max_val}]")
                            unique_vals, counts = np.unique(valid_values, return_counts=True)
                            print(f"Number of unique unmasked values: {len(unique_vals)}")
                        else:
                            print("No valid (unmasked) values found!")

                        # Get vmin/vmax of plot based on data range
                        if masked_band.compressed().size > 0:
                            valid_data = masked_band.compressed()
                            vmin = np.percentile(valid_data, 2)
                            vmax = np.percentile(valid_data, 98)

                            if vmin == vmax:
                                vmin = valid_data.min()
                                vmax = valid_data.max()
                                if vmin == vmax:
                                    vmax = vmin + 1
                        else:
                            vmin, vmax = 0, 1

                        cmap = plt.cm.viridis.copy()
                        cmap.set_bad(color='lightgray')

                        im = axes[row, col].imshow(masked_band, cmap=cmap, vmin=vmin, vmax=vmax)
                        axes[row, col].set_title(f"Band {current_band_num + 1}", fontsize=8)
                        axes[row, col].axis("off")
                        fig.colorbar(im, ax=axes[row, col], fraction=0.046, pad=0.04)
                    else:
                        axes[row, col].axis("off")

                    band_idx += 1

            plt.tight_layout()
            plt.savefig(f"{output_dir}/{base_tiff_path}_{output_basename}{i}_{i+25}.png", dpi=150, bbox_inches='tight')
            plt.close(fig)
            print(f"Saved figure to path: {output_dir}/{base_tiff_path}_{output_basename}{i}_{i+25}.png")

In [10]:
# n_samples = len(tiffs)
# n_bands = 100
# viz_dir = "/explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/viz_datacubes/default_plots"
# plot_data(tiffs, n_samples, n_bands, viz_dir)

# Inference

## Functions

### Helpers

In [27]:
# Helper functions
def min_max_scale_bands(bands):
    """Min-max scale each band to [0, 1]"""
    scaled = np.zeros_like(bands, dtype=np.float32)
    for i in range(bands.shape[0]):
        band = bands[i]
        band_min, band_max = band.min(), band.max()
        if band_max > band_min:
            scaled[i] = (band - band_min) / (band_max - band_min)
        else:
            scaled[i] = band
    return scaled

def extract_images(tiff_paths, num_images_to_extract, bands_per_slice=7, bands_per_image=3, band_order=[2, 0, 1], mean=None, std=None):
    """
    Extract unique spatial images by processing complete band slices.

    Logic:
    - Each 7-band slice represents one spatial image
    - Extract only the first `bands_per_image` bands from each slice
    - Check if those bands have all positive values
    - If yes, keep them; if no, skip the entire slice
    - This prevents spatial duplication

    Args:
        tiff_paths: List of TIFF file paths
        num_images_to_extract: Number of unique spatial images to extract
        bands_per_slice: Total bands per complete image (default: 7)
        bands_per_image: Number of bands to extract from each slice (default: 3)
        band_order: Order to arrange the extracted bands (e.g., [2, 0, 1] for RGB)

    Returns:
        Array of shape (N, H, W, bands_per_image)
    """
    if mean is None or std is None:
        raise ValueError("Mean/std are required to preprocess inputs.")
    else:
        print(f"Extracting images with mean={mean}, std={std}")

    extracted = []

    for tiff_path in tiff_paths:
        if len(extracted) >= num_images_to_extract:
            break

        with xr.open_dataset(tiff_path, engine='rasterio') as ds:
            total_bands = ds.sizes['band']
            num_slices = total_bands // bands_per_slice

            print(f"Processing {tiff_path}: {total_bands} bands, {num_slices} complete {bands_per_slice}-band slices")

            for slice_idx in range(num_slices):
                if len(extracted) >= num_images_to_extract:
                    break

                # Calculate starting band for this 7-band slice
                start_band = slice_idx * bands_per_slice

                # Extract first `bands_per_image` bands from this slice
                slice_data = ds.isel(band=slice(start_band, start_band + bands_per_image))
                data_var = list(slice_data.data_vars)[0]
                image_data = slice_data[data_var].values  # Shape: (bands_per_image, H, W)

                # Check if ALL values in these bands are positive
                NODATA = -3.4028227e+38
                nodata_count = np.sum(image_data == NODATA)
                nodata_percentage = (nodata_count / image_data.size) * 100
                if nodata_count == 0:  # zero tolerance
                    # Reorder bands (e.g., [2, 0, 1] for RGB order)
                    image_data_reordered = image_data[band_order, :, :]

                    # Transpose to (H, W, C) for channels-last format
                    image_data_reordered = np.transpose(image_data_reordered, (1, 2, 0))

                    # Scale inputs to 0, 1
                    image_data_scaled = min_max_scale_bands(image_data_reordered)

                    # Normalize to work with shape
                    mean_reshaped = mean.reshape(1, 1, 3)  # 1 value per band
                    std_reshaped = std.reshape(1, 1, 3)  # 1 value per band
                    print(f"Image shape before norm, after resize: {image_data_reordered.shape}")
                    print(f"Mean, std shapes before norm: {mean_reshaped.shape, std_reshaped.shape}")
                    image_data_norm = (image_data_scaled - mean_reshaped) / std_reshaped

                    extracted.append(image_data_norm)
                    print(f"  ✓ Slice {slice_idx}: bands {start_band}-{start_band + bands_per_image - 1} → valid")
                else:
                    print(f"  ✗ Slice {slice_idx}: bands {start_band}-{start_band + bands_per_image - 1} → skipped (non-positive values)")
                    min_val = np.min(image_data)
                    print(f"     Slice min value: {min_val}, count and percent: {nodata_count}, {nodata_percentage}")

    if len(extracted) < num_images_to_extract:
        raise ValueError(
            f"Requested {num_images_to_extract} valid images but only found "
            f"{len(extracted)} with all positive values across all TIFFs"
        )

    result = np.array(extracted)
    print(f"\nFinal extracted array shape: {result.shape}")  # Should be (N, H, W, 3)

    # Show first few for verification
    for i, elem in enumerate(result[:5]):
        print(f"Image {i} shape: {elem.shape}, min: {elem.min():.2f}, max: {elem.max():.2f}")

    return result


def create_binary_colormap(instance_mask):
    """
    Create a simple binary colormap: 0 -> black, any other value -> red.

    Args:
        instance_mask: (H, W) array with instance IDs (0 = background)

    Returns:
        colored: (H, W, 3) RGB array with values in [0, 1]
    """
    h, w = instance_mask.shape
    colored = np.zeros((h, w, 3), dtype=np.float32)

    # Set all non-zero pixels to red
    mask = instance_mask != 0
    colored[mask] = [1.0, 0.0, 0.0]  # Pure red

    return colored


def save_image_to_tif(i, image, n_bands=7, save_path="image.tif"):
    '''Writes 7-band image to save_path as .tif'''
    driver = gdal.GetDriverByName('GTiff')

    # Create output file with 7 bands
    output_dataset = driver.Create(
        save_path,
        image.shape[2],  # width (columns)
        image.shape[1],  # height (rows)
        n_bands,               # number of bands
        gdal.GDT_Float32
    )

    # Write each band
    for band_num in range(7):
        output_band = output_dataset.GetRasterBand(band_num + 1)  # Band numbering starts at 1
        output_band.WriteArray(image[band_num, :, :])
        output_band.FlushCache()

    print(f"Saved all bands to {save_path}")

    # Close the dataset
    output_dataset = None
    return


def get_tilers_mergers(images_npy, TARGET_SIZE):
    """Gets tilers and mergers used for sliding window inference."""
    print("Creating tilers and mergers...")
    input_tiler = Tiler(
        data_shape=images_npy.shape,
        tile_shape=(TARGET_SIZE[0], TARGET_SIZE[1], 3),
        channel_dimension=-1,
        mode='reflect'
    )
    # Model output should be (304, 304, 1), aka target size
    output_tiler = Tiler(
        data_shape=images_npy.shape,
        tile_shape=(TARGET_SIZE[0], TARGET_SIZE[1], 1),
        channel_dimension=-1,
        mode='reflect'
    )
    output_merger = Merger(tiler=output_tiler, window='triang')
    return input_tiler, output_tiler, output_merger

In [13]:
def sliding_window_inference(images_npy, model, target_size=304, device='cuda', threshold=0.5):
    """
    Perform sliding window inference on large images.
    Merges probabilities then thresholds for smoother results.
    """
    if isinstance(target_size, int):
        target_size = (target_size, target_size)
    
    model.eval()
    
    # Handle single image
    single_image = False
    if images_npy.ndim == 3:
        images_npy = images_npy[np.newaxis, ...]
        single_image = True
    
    all_predictions = []
    all_probabilities = []  # Optional: keep probabilities too
    
    # Process each image
    for idx in range(images_npy.shape[0]):
        print(f"\nProcessing image {idx + 1}/{images_npy.shape[0]}")
        image = images_npy[idx]  # Shape: (512, 512, 3)
        
        # Create INPUT tiler
        input_tiler = Tiler(
            data_shape=image.shape,  # (512, 512, 3)
            tile_shape=(target_size[0], target_size[1], 3),
            channel_dimension=-1,
            mode='reflect'
        )
        
        # Create OUTPUT tiler for SINGLE channel (crater probabilities only)
        output_tiler = Tiler(
            data_shape=(image.shape[0], image.shape[1], 1),  # (512, 512, 1)
            tile_shape=(target_size[0], target_size[1], 1),  # (304, 304, 1)
            channel_dimension=-1,
            mode='reflect'
        )
        
        # Create merger based on OUTPUT tiler
        output_merger = Merger(tiler=output_tiler, window='triang')
        
        print(f"Number of tiles: {len(input_tiler)}")
        
        # Iterate through tiles
        with torch.no_grad():
            for tile_id, tile_batch in input_tiler(image, batch_size=1, progress_bar=True):
                # tile_batch shape: (1, 304, 304, 3)
                tile = tile_batch[0]  # (304, 304, 3)
                
                # Convert to torch tensor and change to channels-first
                tile_tensor = torch.from_numpy(tile).permute(2, 0, 1)  # (3, 304, 304)
                tile_tensor = tile_tensor.unsqueeze(0).float().to(device)  # (1, 3, 304, 304)
                
                # Run inference
                logits = model(tile_tensor)  # Shape: (1, 2, 304, 304)
                
                # Convert to probabilities (same as training eval)
                probs = torch.sigmoid(logits)  # (1, 2, 304, 304)
                
                # Get crater class only (class 1)
                if probs.shape[1] == 2:
                    probs = probs[:, 1:2]  # (1, 1, 304, 304) - crater class
                
                # Convert to numpy and channels-last
                probs_np = probs.cpu().numpy()
                probs_np = np.transpose(probs_np[0], (1, 2, 0))  # (304, 304, 1)
                
                # Add PROBABILITIES to merger (not thresholded yet!)
                actual_tile_id = tile_id * 1  # Since batch_size=1
                output_merger.add(actual_tile_id, probs_np)
        
        # Merge all tiles - this gives smooth probability map
        merged_probs = output_merger.merge(unpad=True)  # (512, 512, 1)
        print(f"Merged probabilities shape: {merged_probs.shape}")
        
        # Threshold to get binary predictions (same as training eval)
        merged_preds = (merged_probs > threshold).astype(np.float32)  # (512, 512, 1)
        merged_preds = merged_preds.squeeze(-1)  # (512, 512) - squeeze channel dim
        
        all_predictions.append(merged_preds)
        all_probabilities.append(merged_probs.squeeze(-1))  # Optional: keep probs
        
        print(f"Final prediction shape: {merged_preds.shape}")
        print(f"Prediction range: [{merged_preds.min():.2f}, {merged_preds.max():.2f}]")
    
    predictions = np.stack(all_predictions, axis=0)
    probabilities = np.stack(all_probabilities, axis=0)
    
    if single_image:
        predictions = predictions[0]
        probabilities = probabilities[0]
    
    return predictions, probabilities  # Return both

### Driver

In [17]:
def visualize_inference_from_tiffs(
    model,
    device,
    tiff_dir,
    mean,  # No longer needed, but kept for compatibility
    std,   # No longer needed, but kept for compatibility
    output_path="inference_test.png",
    n_images=20,
    model_native_size=304,
    tile_overlap=0.25,
    threshold=0.75,
    normalize=True,  # Ignore this parameter
    save_inputs_dir=None, 
    use_sliding=True,
):
    model.eval()

    # Load TIFF files
    tiffs = glob(f"{tiff_dir}/*.tif")
    print(f"Found {len(tiffs)} TIFF files")

    if len(tiffs) == 0:
        raise ValueError(f"No TIFF files found in {tiff_dir}")

    # Load numpy array of 3-band images at full resolution
    print(f"Loading {n_images} images from TIFF files...")
    images_npy = extract_images(
        tiff_paths=tiffs,
        num_images_to_extract=n_images,
        bands_per_slice=5,
        bands_per_image=3,
        band_order=[2, 0, 1], 
        mean=mean, 
        std=std,
    )
    print(f"Extracted images shape: {images_npy.shape}")  # (N, 512, 512, 3)

    # Save inputs if specified
    if save_inputs_dir:
        save_inputs_dir = Path(save_inputs_dir)
        save_inputs_dir.mkdir(exist_ok=True, parents=True)
        for i, image in enumerate(images_npy):
            save_path = save_inputs_dir / f"image_{i}.tif"
            image_transposed = np.transpose(image, (2, 0, 1))
            save_image_to_tif(i, image_transposed, n_bands=3, save_path=str(save_path))

    # Pass raw images, image_processor handles normalization
    print(f"\n{'='*60}")
    print(f"Running inference:")
    print(f"  Input images: {images_npy.shape[1]}×{images_npy.shape[2]}")
    print(f"  Model native resolution: {model_native_size}×{model_native_size}")
    print(f"  Processing with sliding window (overlap={tile_overlap})")
    print(f"{'='*60}\n")

    # Run sliding window inference
    if use_sliding:
        target_size = (model_native_size, model_native_size)
        preds_list, probabilities = sliding_window_inference(
            images_npy, model, target_size=model_native_size, device='cuda', threshold=threshold
        )
    else:  # add support for resizing of inputs then regular pred
        preds_list = None

    print(f"\nGot {len(preds_list)} predictions")
    print(f"Output mask shapes: {[p.shape for p in preds_list[:3]]}")  # Should all be (512, 512)

    # Create visualization
    print("\nCreating visualization...")

    batch_size = min(len(images_npy), 10)  # Limit visualization to 10 images
    fig, axes = plt.subplots(2, batch_size, figsize=(5 * batch_size, 10))

    if batch_size == 1:
        axes = axes.reshape(-1, 1)

    filenames = [f"image_{i}" for i in range(batch_size)]

    for i in range(batch_size):
        img = images_npy[i]  # Original unnormalized image (512, 512, 3)
        pred_mask = preds_list[i]  # (512, 512)

        # Prepare image for display
        img_vis = img.copy()
        # Normalize to [0, 1] for display
        img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min() + 1e-8)

        # Row 0: Original image
        axes[0, i].imshow(img_vis)
        axes[0, i].set_title(
            f"{filenames[i]}",
            fontsize=11,
            fontweight="bold",
        )
        axes[0, i].axis("off")

        # Row 1: Inference with binary colormap
        pred_colored = create_binary_colormap(pred_mask)
        axes[1, i].imshow(pred_colored, vmin=0, vmax=1)
        axes[1, i].set_title(
            f"Inference",
            fontsize=11,
        )
        axes[1, i].axis("off")

    fig.suptitle(
        f"Inference on {images_npy.shape[1]}×{images_npy.shape[2]} Images "
        f"(processed with {model_native_size}×{model_native_size} tiles)\n"
        f"Threshold={threshold}",
        fontsize=14,
        fontweight="bold",
        y=0.98,
    )

    plt.tight_layout()

    # Save figure
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()

    print(f"\nSaved visualization to: {output_path}")

    model.train()

    return images_npy, preds_list

## Call fcn

In [28]:
# Run inference, create viz of inference
n_images = 3
images, predictions = visualize_inference_from_tiffs(
    model=model,
    device=device,
    tiff_dir=INPUT_DIR,
    mean=MEAN,
    std=STD,
    output_path="sem_seg_inf_test_3_19.png",
    n_images=n_images,
    model_native_size=TARGET_SIZE[0],
    tile_overlap=0.25,
    threshold=0.75,
    normalize=True,
    save_inputs_dir=None,
    use_sliding=True,
)

Found 73 TIFF files
Loading 3 images from TIFF files...
Extracting images with mean=[0.32744208 0.32249309 0.302985  ], std=[0.15045737 0.15014801 0.14386101]
Processing /explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/data_cubes/5_band_WAC/Cube-LTM30N_Zoom-5_Tile-1-22.tif: 564 bands, 112 complete 5-band slices
  ✗ Slice 0: bands 0-2 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
  ✗ Slice 1: bands 5-7 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
  ✗ Slice 2: bands 10-12 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
  ✗ Slice 3: bands 15-17 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
  ✗ Slice 4: bands 20-22 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
 

  0%|          | 0/4 [00:00<?, ? batches/s]

Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 1.00]

Processing image 2/3
Number of tiles: 4


  0%|          | 0/4 [00:00<?, ? batches/s]

Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 1.00]

Processing image 3/3
Number of tiles: 4


  0%|          | 0/4 [00:00<?, ? batches/s]

Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 1.00]

Got 3 predictions
Output mask shapes: [(512, 512), (512, 512), (512, 512)]

Creating visualization...

Saved visualization to: sem_seg_inf_test_3_19.png
